# Etapa 2: Modelagem de Tópicos Neural (BERTopic) vs. Clássica (LDA)

Neste notebook, executamos a etapa de **Descoberta Não Supervisionada de Subáreas** do processo KDD para o artigo do **EduComp**, estruturada em três passos:
1. **Geração de Representações Neurais Densas (*Sentence Embeddings*):** Codificação semântica dos resumos da pós-graduação em Ciência da Computação utilizando o modelo multilíngue `paraphrase-multilingual-MiniLM-L12-v2`.
2. **Clusterização Semântica via BERTopic:** Redução de dimensionalidade (**UMAP**) + agrupamento hierárquico por densidade (**HDBSCAN**) + extração vocabular de unigramas e bigramas via **c-TF-IDF**, com refinamento de *outliers*.
3. **Validação e Benchmarking com LDA (*Latent Dirichlet Allocation*):** Comparação empírica entre os tópicos descobertos pelo BERTopic e a linha de base probabilística clássica quanto à **Coerência Semântica (NPMI)** e **Diversidade Vocabular**, avaliada pelo teste não-paramétrico de **Mann-Whitney U**.

In [1]:
import os
import ast
import json
import time
import itertools
import pandas as pd
import numpy as np
import torch
from scipy import stats

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

REPO = '../data'
OUT = '../data/resultados'
os.makedirs(OUT, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de execução PyTorch detectado: {device.upper()}")

Dispositivo de execução PyTorch detectado: CPU


## 1. Carregamento da Base Pré-processada
Lemos a base resultante da Etapa 1 (`dados_capes_nlp.csv.gz`). Para manter a consistência metodológica dos 37 clusters temáticos consolidados (período 2010–2023), filtramos os documentos pré-processados desse intervalo (os registros recém-lançados de 2024 são integrados diretamente na Etapa 3 via rotulagem léxica).

In [2]:
caminho_entrada = f'{REPO}/dados_capes_nlp.csv.gz'
df = pd.read_csv(caminho_entrada, compression='gzip')

df['AN_BASE'] = df['AN_BASE'].astype(int)
df = df[df['AN_BASE'] <= 2023].dropna(subset=['DS_RESUMO', 'RESUMO_LIMPO']).reset_index(drop=True)

print(f"Total de teses e dissertações válidas para modelagem de tópicos: {len(df)}")
print(f"Período coberto: {df['AN_BASE'].min()} a {df['AN_BASE'].max()}")
df[['AN_BASE', 'SG_ENTIDADE_ENSINO', 'NM_REGIAO', 'NM_GRAU_ACADEMICO', 'NM_PRODUCAO', 'RESUMO_LIMPO']].head(3)

Total de teses e dissertações válidas para modelagem de tópicos: 22130
Período coberto: 2010 a 2023


,AN_BASE,SG_ENTIDADE_ENSINO,NM_REGIAO,NM_GRAU_ACADEMICO,NM_PRODUCAO,RESUMO_LIMPO
0,2012,UFAM,NORTE,MESTRADO,Sistema Inteligente de Recomendação Baseado n...,disseminação curso educação distância frequent...
1,2012,UFAM,NORTE,MESTRADO,Alinhamento Espaço-Temporal em Sistemas Multis...,utilização diferentes sensor aprimoramento inf...
2,2012,UFAM,NORTE,MESTRADO,''UBICUA: Uma Técnica de Inspeção de Usabilida...,crescente interesse tecnologia móvel evolutivo...


## 2. Geração e Sincronização de Embeddings Semânticos
Verificamos se a matriz de *embeddings* salva em disco (`embeddings_capes.npy`) possui exatamente o mesmo número de registros da base ativa; caso contrário, recalculamos os vetores automaticamente.

In [3]:
caminho_embeddings = f'{REPO}/embeddings_capes.npy'
modelo_embedding_nome = 'paraphrase-multilingual-MiniLM-L12-v2'
embedding_model = SentenceTransformer(modelo_embedding_nome, device=device)

documentos = df['RESUMO_LIMPO'].tolist()

recalcular = True
if os.path.exists(caminho_embeddings):
    embeddings = np.load(caminho_embeddings)
    if embeddings.shape[0] == len(documentos):
        print("Carregando embeddings pré-calculados do disco...")
        recalcular = False
    else:
        print(f"Inconsistência detectada: {len(documentos)} docs vs {embeddings.shape[0]} embeddings salvos. Recalculando...")

if recalcular:
    print("Calculando embeddings dos resumos (isso pode levar alguns minutos)...")
    embeddings = embedding_model.encode(
        documentos, 
        show_progress_bar=True, 
        batch_size=64,
        convert_to_numpy=True
    )
    np.save(caminho_embeddings, embeddings)
    print(f"Embeddings salvos com sucesso em {caminho_embeddings}!")

print(f"Formato da matriz de embeddings: {embeddings.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Inconsistência detectada: 22130 docs vs 23851 embeddings salvos. Recalculando...
Calculando embeddings dos resumos (isso pode levar alguns minutos)...


Batches:   0%|          | 0/346 [00:00<?, ?it/s]

Embeddings salvos com sucesso em ../data/embeddings_capes.npy!
Formato da matriz de embeddings: (22130, 384)


## 3. Treinamento do BERTopic e Exportação dos Tópicos
Caso a base já tenha sido rotulada e salva anteriormente com os 37 tópicos (`dados_capes_topicos_ia.csv.gz`), carregamos os tópicos existentes para preservar a reprodutibilidade exata das tabelas do artigo; caso contrário, executamos o pipeline completo do BERTopic.

In [4]:
caminho_saida_topicos = f'{REPO}/dados_capes_topicos_ia.csv.gz'
caminho_info_topicos = f'{REPO}/tabela_resumo_topicos_bertopic.csv.gz'

if os.path.exists(caminho_saida_topicos) and os.path.exists(caminho_info_topicos):
    print("Carregando modelo de tópicos BERTopic consolidado do disco...")
    df = pd.read_csv(caminho_saida_topicos, compression='gzip')
    info_topicos = pd.read_csv(caminho_info_topicos, compression='gzip')
else:
    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
    hdbscan_model = HDBSCAN(min_cluster_size=80, min_samples=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
    vectorizer_model = CountVectorizer(ngram_range=(1, 2), min_df=10)

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        nr_topics=38,
        top_n_words=10,
        verbose=True
    )

    print("Treinando o modelo BERTopic...")
    topicos, probabilidades = topic_model.fit_transform(documentos, embeddings)

    print("Refinando outliers (tópico -1) para melhorar a cobertura temática...")
    novos_topicos = topic_model.reduce_outliers(documentos, topicos, strategy="embeddings", embeddings=embeddings)
    topic_model.update_topics(documentos, topics=novos_topicos, vectorizer_model=vectorizer_model)

    df['TOPICO_ID'] = novos_topicos
    info_topicos = topic_model.get_topic_info()
    
    mapa_nomes = dict(zip(info_topicos['Topic'], info_topicos['Name']))
    df['TOPICO_NOME'] = df['TOPICO_ID'].map(mapa_nomes)
    
    df.to_csv(caminho_saida_topicos, index=False, encoding='utf-8', compression='gzip')
    info_topicos.to_csv(caminho_info_topicos, index=False, encoding='utf-8', compression='gzip')

info_validos = info_topicos[info_topicos['Topic'] != -1].reset_index(drop=True)
print(f"Total de tópicos temáticos ativos: {len(info_validos)}")
display(info_validos[['Topic', 'Count', 'Name', 'Representation']].head(15))

Carregando modelo de tópicos BERTopic consolidado do disco...
Total de tópicos temáticos ativos: 16


,Topic,Count,Name,Representation
0,0,18276,0_modelo_rede_software_técnica,"['modelo', 'rede', 'software', 'técnica', 'inf..."
1,1,1031,1_saúde_paciente_doença_modelo,"['saúde', 'paciente', 'doença', 'modelo', 'méd..."
2,2,628,2_sequência_modelo_estrutura_computacional,"['sequência', 'modelo', 'estrutura', 'computac..."
3,3,423,3_energia_elétrico_consumo_consumo energia,"['energia', 'elétrico', 'consumo', 'consumo en..."
4,4,387,4_robôs_ambiente_navegação_robótico,"['robôs', 'ambiente', 'navegação', 'robótico',..."
5,5,337,5_sinal_voz_áudio_reconhecimento,"['sinal', 'voz', 'áudio', 'reconhecimento', 'u..."
6,6,398,6_ensino_aluno_curso_educação,"['ensino', 'aluno', 'curso', 'educação', 'estu..."
7,7,245,7_imagem_diagnóstico_paciente_segmentação,"['imagem', 'diagnóstico', 'paciente', 'segment..."
8,8,273,8_veículo_aéreo_rede_modelo,"['veículo', 'aéreo', 'rede', 'modelo', 'simula..."
9,9,268,9_sensor_rede_rede sensor_fio,"['sensor', 'rede', 'rede sensor', 'fio', 'prot..."


## 4. Benchmarking Não Supervisionado: BERTopic vs. LDA (NPMI e Diversidade)
Treinamos um modelo **LDA (*Latent Dirichlet Allocation*)** com o mesmo número de tópicos ($K$) sobre os resumos limpos e comparamos a qualidade intrínseca dos agrupamentos usando:
- **Coerência NPMI (*Normalized Pointwise Mutual Information*):** Mede a co-ocorrência real das 10 palavras mais representativas de cada tópico nos documentos.
- **Diversidade de Tópicos:** Proporção de termos únicos entre as *top-10* palavras de todos os tópicos (quanto mais próximo de $1{,}0$, menor a redundância entre clusters).

In [5]:
docs_lda = df['RESUMO_LIMPO'].fillna('').tolist()
top_bt = [ast.literal_eval(r)[:10] if isinstance(r, str) else r[:10] for r in info_validos['Representation']]
K = len(top_bt)

# 1. Treinamento do LDA com o mesmo K
cv = CountVectorizer(min_df=10, max_df=0.5, max_features=10000)
X_lda = cv.fit_transform(docs_lda)
voc = np.array(cv.get_feature_names_out())

t0 = time.time()
lda = LatentDirichletAllocation(
    n_components=K, learning_method='online', batch_size=1024, max_iter=15,
    random_state=42, n_jobs=-1
)
theta = lda.fit_transform(X_lda)
print(f"LDA (K={K}) treinado em {time.time()-t0:.1f}s | Perplexidade: {lda.perplexity(X_lda):.0f}")
top_lda = [list(voc[np.argsort(c)[::-1][:10]]) for c in lda.components_]

# 2. Matriz binária de co-ocorrência para cálculo de NPMI
cvb = CountVectorizer(ngram_range=(1, 2), min_df=5, binary=True)
B = cvb.fit_transform(docs_lda).tocsc()
idx_vocab = {w: i for i, w in enumerate(cvb.get_feature_names_out())}
N_docs = B.shape[0]
df_w = np.asarray(B.sum(0)).ravel()

def npmi_topico(palavras):
    ids = [idx_vocab[w] for w in palavras if w in idx_vocab]
    vals = []
    for a, b in itertools.combinations(ids, 2):
        co = B[:, a].multiply(B[:, b]).sum()
        if co == 0:
            vals.append(-1.0)
            continue
        pab = co / N_docs; pa = df_w[a] / N_docs; pb = df_w[b] / N_docs
        vals.append(np.log(pab / (pa * pb)) / -np.log(pab))
    return np.mean(vals) if vals else np.nan

def diversidade(tops):
    todas = [w for t in tops for w in t]
    return len(set(todas)) / len(todas)

npmi_bt = [npmi_topico(t) for t in top_bt]
npmi_lda = [npmi_topico(t) for t in top_lda]

coer = {
    'BERTopic': {'npmi_medio': float(np.nanmean(npmi_bt)), 'npmi_dp': float(np.nanstd(npmi_bt)), 'diversidade': diversidade(top_bt), 'k': len(top_bt)},
    'LDA': {'npmi_medio': float(np.nanmean(npmi_lda)), 'npmi_dp': float(np.nanstd(npmi_lda)), 'diversidade': diversidade(top_lda), 'k': K},
    'mannwhitney_p': float(stats.mannwhitneyu(npmi_bt, npmi_lda).pvalue),
    'lda_perplexidade': float(lda.perplexity(X_lda))
}

# Tendências anuais dos tópicos LDA
th = pd.DataFrame(theta)
th['ano'] = df['AN_BASE'].values
prop_lda = th.groupby('ano').mean() * 100
linhas_lda = []
for k in range(K):
    lr = stats.linregress(prop_lda.index.values, prop_lda[k].values)
    linhas_lda.append({
        'topico_lda': k, 'palavras': ', '.join(top_lda[k][:6]), 
        'inclinacao_pp_ano': lr.slope, 'p': lr.pvalue,
        'ini_2010_12': prop_lda.loc[2010:2012, k].mean(), 
        'fim_2021_23': prop_lda.loc[2021:2023, k].mean()
    })
tl = pd.DataFrame(linhas_lda).sort_values('inclinacao_pp_ano', ascending=False).reset_index(drop=True)
tl.to_csv(f'{OUT}/tendencias_lda.csv', index=False)
json.dump(coer, open(f'{OUT}/resultados_lda.json', 'w'), ensure_ascii=False, indent=1)

print(f"Comparação BERTopic vs. LDA (Mann-Whitney U p-valor = {coer['mannwhitney_p']:.2e}):")
display(pd.DataFrame([coer['BERTopic'], coer['LDA']], index=['BERTopic (Neural)', 'LDA (Clássico)']).round(4))

LDA (K=16) treinado em 92.0s | Perplexidade: 1552
Comparação BERTopic vs. LDA (Mann-Whitney U p-valor = 3.56e-01):


,npmi_medio,npmi_dp,diversidade,k
BERTopic (Neural),0.1619,0.0809,0.7375,16
LDA (Clássico),0.1646,0.1804,0.8188,16
